# Analysis-ready: a DataFrame becomes a track

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cmdcolin/jbrowse-anywidget/blob/main/examples/02_dataframe_analysis.ipynb)

The point of a notebook genome browser is to **see the result of a computation in genomic context**. `add_features` turns a pandas DataFrame straight into a track — no file written, no server.

In [ ]:
# Install only if not already available (e.g. in Colab). The GitHub install
# needs no JS toolchain — the built widget bundle is committed in the repo. A
# local editable install is used as-is. (Swap to `jbrowse-anywidget` once it's
# published to PyPI.)
try:
    import jbrowse_anywidget  # noqa: F401
except ImportError:
    %pip install -q "jbrowse-anywidget @ git+https://github.com/cmdcolin/jbrowse-anywidget" pandas numpy

# Colab requires this to render third-party (anywidget) widgets:
try:
    from google.colab import output

    output.enable_custom_widget_manager()
except ImportError:
    pass

## Compute something

A stand-in analysis: sliding-window mean of a synthetic signal, producing scored intervals. Swap this for your real pipeline output — peaks, coverage, methylation — as long as it has `refName`/`chrom`, `start`, `end` columns.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(0)
start = 29_838_000
positions = np.arange(start, start + 2000)
signal = rng.normal(size=positions.size).cumsum()

win = 100
rows = []
for i in range(0, positions.size - win, win):
    rows.append(
        {
            "chrom": "10",
            "start": int(positions[i]),
            "end": int(positions[i + win]),
            "score": float(signal[i : i + win].mean()),
        }
    )

windows = pd.DataFrame(rows)
windows.head()

## Visualize it in genomic context

In [ ]:
from jbrowse_anywidget import LinearGenomeView, make_assembly

hg38 = make_assembly(
    "hg38",
    "https://jbrowse.org/genomes/GRCh38/fasta/hg38.prefix.fa.gz",
    aliases=["GRCh38"],
)
view = LinearGenomeView(assembly=hg38, location="10:29,838,000..29,840,000")
view.add_features(windows, name="windowed mean")
view

Every column beyond `refName`/`start`/`end` is carried onto the feature, so `score` (and any annotations you add) show up in the feature details and can drive rendering.

## Color by a computed value

A feature's own columns are addressable from a [jexl](https://jbrowse.org/jb2/docs/config_guides/customizing_feature_colors/) color expression, so the track can encode `score` directly — high windows in red, low in blue.

In [ ]:
colored = LinearGenomeView(
    assembly=hg38, location="10:29,838,000..29,840,000"
)
colored.add_features(
    windows,
    name="windowed mean (colored)",
    color="jexl:get(feature,'score') > 0 ? '#c62828' : '#1565c0'",
)
colored